# **Modelo 4_ETL1 — Leave-One-Attack-Out Cross-Validation (LOGOCV)**

Dataset: **dataset_caracteristicas_train_V1_ALL.csv**

## OBJETIVO DEL NOTEBOOK

Implementar una **validación cruzada Leave-One-Group-Out (LOGOCV)** donde cada uno de los 6 algoritmos de ataque (A01–A06) actúa como conjunto de test una vez. Esto permite medir si el modelo es capaz de **generalizar ante ataques que nunca ha visto** durante el entrenamiento.

### Diferencia clave respecto a modelos anteriores:
| Modelo | Estrategia | Limitación |
|--------|-----------|------------|
| **1 y 2** | Split aleatorio estratificado | Test contiene muestras de todos los ataques (fuga de información) |
| **3** | Leave-One-Group-Out (1 vez, A05-A06) | Solo una iteración, resultado no estadísticamente robusto |
| **4 (este)** | LOGOCV completo (6 iteraciones) | Estimación estadísticamente sólida con media y desviación estándar |

### Configuración:
- **Algoritmo:** XGBoost optimizado (`XGB_OPTIMAL_PARAMS_M1`)
- **Features:** 13 features óptimas (Permutation Importance, Modelo 1)
- **Folds:** 6 — cada ataque (A01–A06) queda fuera del entrenamiento una vez
- **Balance:** Undersampling 1:1 en train y test

In [1]:
# ============================================================
# BLOQUE 1: Importaciones y Configuración Global
# ============================================================

import numpy as np
import pandas as pd
import warnings
import os
import glob as glob_module
warnings.filterwarnings('ignore')

from xgboost import XGBClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score
)

# ── Reproducibilidad ─────────────────────────────────────────────────────────
RANDOM_STATE = 42

# ── Rutas de datos ────────────────────────────────────────────────────────────
DATA_TRAIN_PATH = '../../Obtencion_Metricas/dataset_caracteristicas_train_V1_ALL.csv'
DATA_EVAL_PATH  = '../../Obtencion_Metricas/dataset_caracteristicas_eval.csv'
TEST_VOCES_DIR  = '../../test_voces/'

# ── Lista de ataques para el bucle LOGOCV ────────────────────────────────────
ATTACKS = ['A01', 'A02', 'A03', 'A04', 'A05', 'A06']

# ── Hiperparámetros óptimos de XGBoost (RandomizedSearchCV, Modelo 1, N=4000) ─
XGB_OPTIMAL_PARAMS_M1 = {
    'objective'        : 'binary:logistic',
    'eval_metric'      : 'logloss',
    'use_label_encoder': False,
    'random_state'     : RANDOM_STATE,
    'n_jobs'           : -1,
    'n_estimators'     : 620,
    'max_depth'        : 7,
    'learning_rate'    : 0.09591931665418388,
    'subsample'        : 0.602208846849441,
    'colsample_bytree' : 0.6379995910112717,
    'reg_alpha'        : 0.7722447692966574,
    'reg_lambda'       : 1.3942205669037757,
    'min_child_weight' : 1,
}

# ── 13 features óptimas (Permutation Importance sobre XGBoost, Modelo 1) ────
FINAL_FEATURES = [
    'signal_mean', 'mfcc_9_std',  'mfcc_6_mean', 'mfcc_7_mean',
    'mfcc_4_std',  'mfcc_5_mean', 'mfcc_13_mean', 'mfcc_2_std',
    'mfcc_8_std',  'mfcc_3_mean', 'mfcc_10_std',  'mfcc_9_mean',
    'mfcc_11_mean'
]

# ── Tamaño de conjuntos balanceados (consistente con Modelo 3) ───────────────
N_PER_CLASS_TRAIN = 2000   # 2000 spoof + 2000 bonafide = 4000 por fold

# ── Cargar dataset de train ──────────────────────────────────────────────────
df = pd.read_csv(DATA_TRAIN_PATH)

print(f'Dataset cargado        : {len(df):,} registros, {len(df.columns)} columnas')
print(f'Features seleccionadas : {len(FINAL_FEATURES)} de {len(df.columns) - 3}')
print(f'Features               : {FINAL_FEATURES}')
print()
print('Distribución por attack_id:')
print(df['attack_id'].value_counts().sort_index())
print()
print('Distribución por label:')
print(df['label'].value_counts())
print()
print(f'Configuración LOGOCV: {len(ATTACKS)} folds — cada ataque (A01-A06) queda fuera una vez')
print(f'Train por fold      : {N_PER_CLASS_TRAIN*2:,} muestras ({N_PER_CLASS_TRAIN:,} spoof + {N_PER_CLASS_TRAIN:,} bonafide)')

Dataset cargado        : 25,380 registros, 37 columnas
Features seleccionadas : 13 de 34
Features               : ['signal_mean', 'mfcc_9_std', 'mfcc_6_mean', 'mfcc_7_mean', 'mfcc_4_std', 'mfcc_5_mean', 'mfcc_13_mean', 'mfcc_2_std', 'mfcc_8_std', 'mfcc_3_mean', 'mfcc_10_std', 'mfcc_9_mean', 'mfcc_11_mean']

Distribución por attack_id:
attack_id
-      2580
A01    3800
A02    3800
A03    3800
A04    3800
A05    3800
A06    3800
Name: count, dtype: int64

Distribución por label:
label
spoof       22800
bonafide     2580
Name: count, dtype: int64

Configuración LOGOCV: 6 folds — cada ataque (A01-A06) queda fuera una vez
Train por fold      : 4,000 muestras (2,000 spoof + 2,000 bonafide)


In [2]:
# ============================================================
# BLOQUE 2: Bucle Leave-One-Attack-Out Cross-Validation
# ============================================================
# En cada iteración i, el ataque ATTACKS[i] queda fuera del entrenamiento
# y actúa como conjunto de test ("ataque nunca visto").
#
# Estrategia de partición:
#   TEST  → todos los spoof del ataque excluido + slice de bonafide (1/6 del total)
#   TRAIN → spoof de los 5 ataques restantes   + bonafide restante  (5/6 del total)
#   Balance → undersampling 1:1 en ambos conjuntos

df_bonafide = df[df['label'] == 'bonafide'].copy()
df_spoof    = df[df['label'] == 'spoof'].copy()

# Mezclar bonafide una sola vez con semilla fija para que cada fold
# reciba un slice exclusivo y no haya solapamiento entre train y test
df_bonafide = df_bonafide.sample(frac=1, random_state=RANDOM_STATE).reset_index(drop=True)

n_bon_total    = len(df_bonafide)
n_bon_per_fold = n_bon_total // len(ATTACKS)   # ~430 muestras bonafide por fold

fold_results = []

for i, fold_attack in enumerate(ATTACKS):

    print(f'\n{"="*65}')
    print(f'  FOLD {i+1}/{len(ATTACKS)}  |  Ataque excluido del TRAIN: {fold_attack}')
    print(f'{"="*65}')

    # ── TEST: todos los spoof del ataque excluido + slice de bonafide ────────
    test_spoof_pool = df_spoof[df_spoof['attack_id'] == fold_attack].copy()

    bon_start = i * n_bon_per_fold
    bon_end   = (bon_start + n_bon_per_fold
                 if i < (len(ATTACKS) - 1) else n_bon_total)   # último fold toma el resto
    test_bon_pool = df_bonafide.iloc[bon_start:bon_end].copy()

    # Balancear test (1:1): limitado por la clase minoritaria del fold
    n_test_per_class = min(len(test_spoof_pool), len(test_bon_pool))
    test_spoof = test_spoof_pool.sample(n=n_test_per_class, random_state=RANDOM_STATE)
    test_bon   = test_bon_pool.sample(n=n_test_per_class,   random_state=RANDOM_STATE)
    df_test    = (pd.concat([test_spoof, test_bon])
                    .sample(frac=1, random_state=RANDOM_STATE)
                    .reset_index(drop=True))

    # ── TRAIN: los 5 ataques restantes + bonafide no usado en test ───────────
    train_spoof_pool = df_spoof[df_spoof['attack_id'] != fold_attack].copy()

    bon_idx_remaining = list(range(0, bon_start)) + list(range(bon_end, n_bon_total))
    train_bon_pool    = df_bonafide.iloc[bon_idx_remaining].copy()

    # Undersampling balanceado al N_PER_CLASS_TRAIN
    n_train_per_class = min(N_PER_CLASS_TRAIN, len(train_spoof_pool), len(train_bon_pool))
    train_spoof = train_spoof_pool.sample(n=n_train_per_class, random_state=RANDOM_STATE)
    train_bon   = train_bon_pool.sample(n=n_train_per_class,   random_state=RANDOM_STATE)
    df_train    = (pd.concat([train_spoof, train_bon])
                     .sample(frac=1, random_state=RANDOM_STATE)
                     .reset_index(drop=True))

    # ── Información del fold ─────────────────────────────────────────────────
    print(f'  TRAIN → Spoof ({fold_attack} excluido): {len(train_spoof):,} '
          f'| Bonafide: {len(train_bon):,} | Total: {len(df_train):,}')
    print(f'  TEST  → Spoof ({fold_attack}): {len(test_spoof):,} '
          f'| Bonafide: {len(test_bon):,} | Total: {len(df_test):,}')

    ataques_en_train = (df_train[df_train['label'] == 'spoof']['attack_id']
                        .value_counts().sort_index().to_dict())
    print(f'  Distribución spoof en TRAIN: {ataques_en_train}')

    # ── Preparar matrices X, y ───────────────────────────────────────────────
    X_train = df_train[FINAL_FEATURES].values
    y_train = (df_train['label'] == 'spoof').astype(int).values
    X_test  = df_test[FINAL_FEATURES].values
    y_test  = (df_test['label'] == 'spoof').astype(int).values

    # ── Entrenar XGBoost con hiperparámetros óptimos ─────────────────────────
    print(f'  Entrenando XGBoost (n_estimators={XGB_OPTIMAL_PARAMS_M1["n_estimators"]}, '
          f'max_depth={XGB_OPTIMAL_PARAMS_M1["max_depth"]})...')
    model = XGBClassifier(**XGB_OPTIMAL_PARAMS_M1)
    model.fit(X_train, y_train)

    # ── Predicción y métricas del fold ───────────────────────────────────────
    y_pred = model.predict(X_test)

    acc  = accuracy_score(y_test,  y_pred)
    prec = precision_score(y_test, y_pred)
    rec  = recall_score(y_test,    y_pred)
    f1   = f1_score(y_test,        y_pred)

    print(f'  Resultado → Accuracy: {acc:.4f} | Precision: {prec:.4f} '
          f'| Recall: {rec:.4f} | F1: {f1:.4f}')

    fold_results.append({
        'fold'     : f'Fold {fold_attack}',
        'accuracy' : acc,
        'precision': prec,
        'recall'   : rec,
        'f1'       : f1,
    })

print(f'\n{"="*65}')
print(f'  LOGOCV completado — {len(fold_results)}/{len(ATTACKS)} folds procesados correctamente')
print(f'{"="*65}')


  FOLD 1/6  |  Ataque excluido del TRAIN: A01
  TRAIN → Spoof (A01 excluido): 2,000 | Bonafide: 2,000 | Total: 4,000
  TEST  → Spoof (A01): 430 | Bonafide: 430 | Total: 860
  Distribución spoof en TRAIN: {'A02': 359, 'A03': 404, 'A04': 416, 'A05': 419, 'A06': 402}
  Entrenando XGBoost (n_estimators=620, max_depth=7)...
  Resultado → Accuracy: 0.9395 | Precision: 0.9200 | Recall: 0.9628 | F1: 0.9409

  FOLD 2/6  |  Ataque excluido del TRAIN: A02
  TRAIN → Spoof (A02 excluido): 2,000 | Bonafide: 2,000 | Total: 4,000
  TEST  → Spoof (A02): 430 | Bonafide: 430 | Total: 860
  Distribución spoof en TRAIN: {'A01': 359, 'A03': 404, 'A04': 416, 'A05': 419, 'A06': 402}
  Entrenando XGBoost (n_estimators=620, max_depth=7)...
  Resultado → Accuracy: 0.9488 | Precision: 0.9270 | Recall: 0.9744 | F1: 0.9501

  FOLD 3/6  |  Ataque excluido del TRAIN: A03
  TRAIN → Spoof (A03 excluido): 2,000 | Bonafide: 2,000 | Total: 4,000
  TEST  → Spoof (A03): 430 | Bonafide: 430 | Total: 860
  Distribución spoof

In [3]:
# ============================================================
# BLOQUE 3: Tabla resumen de resultados LOGOCV
# ============================================================

df_results = pd.DataFrame(fold_results)

# ── Estadísticas agregadas ───────────────────────────────────────────────────
means = df_results[['accuracy', 'precision', 'recall', 'f1']].mean()
stds  = df_results[['accuracy', 'precision', 'recall', 'f1']].std(ddof=1)  # std muestral

# ── Tabla en formato caja ────────────────────────────────────────────────────
print('\u250c\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u252c\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u252c\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u252c\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u252c\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2510')
print('\u2502  Fold    \u2502 Accuracy \u2502 Precision \u2502 Recall \u2502 F1-Score \u2502')
print('\u251c\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u253c\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u253c\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u253c\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u253c\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2524')

for r in fold_results:
    print(f"\u2502 {r['fold']:<8} \u2502  {r['accuracy']:.4f}  \u2502  {r['precision']:.4f}   \u2502 {r['recall']:.4f} \u2502  {r['f1']:.4f}  \u2502")

print('\u251c\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u253c\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u253c\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u253c\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u253c\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2524')
print(f"\u2502  MEDIA   \u2502  {means['accuracy']:.4f}  \u2502  {means['precision']:.4f}   \u2502 {means['recall']:.4f} \u2502  {means['f1']:.4f}  \u2502")
print(f"\u2502  STD DEV \u2502  {stds['accuracy']:.4f}  \u2502  {stds['precision']:.4f}   \u2502 {stds['recall']:.4f} \u2502  {stds['f1']:.4f}  \u2502")
print('\u2514\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2534\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2534\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2534\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2534\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2518')

# ── Interpretación automática ────────────────────────────────────────────────
print()
print('Interpretación rápida:')

f1_mean = means['f1']
f1_std  = stds['f1']

if f1_mean >= 0.90:
    gen_nivel = 'ALTO — el modelo generaliza bien ante ataques desconocidos'
elif f1_mean >= 0.75:
    gen_nivel = 'MODERADO — generalización parcial, sensible a algunos ataques'
else:
    gen_nivel = 'BAJO — el modelo depende fuertemente de los ataques vistos'

if f1_std > 0.08:
    var_nivel = 'ALTA variabilidad — la eficacia depende críticamente del tipo de ataque'
elif f1_std > 0.04:
    var_nivel = 'VARIABILIDAD MODERADA — algunos ataques son más difíciles de detectar'
else:
    var_nivel = 'BAJA variabilidad — el modelo generaliza de forma estable entre ataques'

print(f'  F1 media  : {f1_mean:.4f} → {gen_nivel}')
print(f'  F1 std dev: {f1_std:.4f} → {var_nivel}')

# ── Mejor y peor fold ────────────────────────────────────────────────────────
best_fold  = df_results.loc[df_results['f1'].idxmax()]
worst_fold = df_results.loc[df_results['f1'].idxmin()]
print(f'  Mejor fold : {best_fold["fold"]}  (F1 = {best_fold["f1"]:.4f})')
print(f'  Peor fold  : {worst_fold["fold"]}  (F1 = {worst_fold["f1"]:.4f})')

┌──────────┬──────────┬───────────┬────────┬──────────┐
│  Fold    │ Accuracy │ Precision │ Recall │ F1-Score │
├──────────┼──────────┼───────────┼────────┼──────────┤
│ Fold A01 │  0.9395  │  0.9200   │ 0.9628 │  0.9409  │
│ Fold A02 │  0.9488  │  0.9270   │ 0.9744 │  0.9501  │
│ Fold A03 │  0.6965  │  0.8658   │ 0.4651 │  0.6051  │
│ Fold A04 │  0.7570  │  0.8932   │ 0.5837 │  0.7060  │
│ Fold A05 │  0.8233  │  0.8949   │ 0.7326 │  0.8056  │
│ Fold A06 │  0.5384  │  0.7089   │ 0.1302 │  0.2200  │
├──────────┼──────────┼───────────┼────────┼──────────┤
│  MEDIA   │  0.7839  │  0.8683   │ 0.6415 │  0.7046  │
│  STD DEV │  0.1560  │  0.0811   │ 0.3219 │  0.2724  │
└──────────┴──────────┴───────────┴────────┴──────────┘

Interpretación rápida:
  F1 media  : 0.7046 → BAJO — el modelo depende fuertemente de los ataques vistos
  F1 std dev: 0.2724 → ALTA variabilidad — la eficacia depende críticamente del tipo de ataque
  Mejor fold : Fold A02  (F1 = 0.9501)
  Peor fold  : Fold A06  (F1 = 0

In [4]:
# ============================================================
# BLOQUE 4: Modelo Final — Entrenamiento completo y evaluación externa
# ============================================================
# Una vez validado con LOGOCV, entrenamos el modelo final con TODOS
# los datos de train (A01-A06) y lo evaluamos sobre:
#   (a) dataset_caracteristicas_eval.csv  → ataques A07-A19, nunca vistos
#   (b) test_voces/                       → voces reales grabadas por el equipo

import librosa

# ── 4.1 Entrenar modelo final con todos los datos de train (A01-A06) ────────
print('=' * 65)
print('PARTE 1: Entrenamiento del Modelo Final (A01-A06 completos)')
print('=' * 65)

df_spoof_final    = df[df['label'] == 'spoof'].copy()
df_bonafide_final = df[df['label'] == 'bonafide'].copy()

# Usar todos los bonafide disponibles y balancear spoof al mismo número
n_final_per_class = min(len(df_bonafide_final), len(df_spoof_final))
train_spoof_f    = df_spoof_final.sample(n=n_final_per_class, random_state=RANDOM_STATE)
train_bonafide_f = df_bonafide_final.sample(n=n_final_per_class, random_state=RANDOM_STATE)
df_train_final   = (pd.concat([train_spoof_f, train_bonafide_f])
                      .sample(frac=1, random_state=RANDOM_STATE)
                      .reset_index(drop=True))

X_final = df_train_final[FINAL_FEATURES].values
y_final = (df_train_final['label'] == 'spoof').astype(int).values

model_final = XGBClassifier(**XGB_OPTIMAL_PARAMS_M1)
model_final.fit(X_final, y_final)

print(f'Modelo final entrenado: {len(df_train_final):,} muestras '
      f'({len(train_spoof_f):,} spoof + {len(train_bonafide_f):,} bonafide)')
print(f'Distribución de ataques en train final:')
print(df_train_final[df_train_final['label'] == 'spoof']['attack_id'].value_counts().sort_index())

# ── 4.2 Evaluación sobre dataset EVAL (ataques A07-A19, nunca vistos) ───────
print()
print('=' * 65)
print('PARTE 2: Evaluación sobre dataset EVAL (ataques A07-A19)')
print('=' * 65)

if not os.path.exists(DATA_EVAL_PATH):
    print(f'AVISO: No se encontró {DATA_EVAL_PATH}')
    print('  Ejecuta primero obtencion_metricas_V1.ipynb (configurado para eval)')
    print('  y vuelve a correr esta celda.')
else:
    df_eval = pd.read_csv(DATA_EVAL_PATH)
    print(f'Dataset eval cargado: {len(df_eval):,} registros')
    print(f'Distribución por clase:')
    print(df_eval['label'].value_counts())
    print(f'Ataques presentes: {sorted(df_eval["attack_id"].unique())}')

    feats_faltantes = [f for f in FINAL_FEATURES if f not in df_eval.columns]
    if feats_faltantes:
        print(f'ERROR: Columnas faltantes en eval CSV: {feats_faltantes}')
    else:
        X_eval = df_eval[FINAL_FEATURES].values
        y_eval = (df_eval['label'] == 'spoof').astype(int).values
        y_pred_eval = model_final.predict(X_eval)

        print()
        print('Métricas del modelo final sobre EVAL (ataques A07-A19 — nunca vistos):')
        print(f'  Accuracy : {accuracy_score(y_eval, y_pred_eval):.4f}')
        print(f'  Precision: {precision_score(y_eval, y_pred_eval):.4f}')
        print(f'  Recall   : {recall_score(y_eval, y_pred_eval):.4f}')
        print(f'  F1-Score : {f1_score(y_eval, y_pred_eval):.4f}')

# ── 4.3 Evaluación sobre test_voces/ (voces reales del equipo) ──────────────
print()
print('=' * 65)
print('PARTE 3: Evaluación sobre test_voces/ (voces reales del equipo)')
print('=' * 65)

def extract_features_13(file_path):
    """Extrae las 13 features óptimas de un archivo de audio usando librosa.
    Misma lógica que extract_features() en obtencion_metricas_V1.ipynb.
    """
    try:
        y_audio, sr = librosa.load(file_path, sr=None)
    except Exception as e:
        print(f'  AVISO: Error cargando {os.path.basename(file_path)}: {e}')
        return None

    features = {}
    features['signal_mean'] = float(np.mean(y_audio))

    mfccs = librosa.feature.mfcc(y=y_audio, sr=sr, n_mfcc=13)
    for i in range(1, 14):
        features[f'mfcc_{i}_mean'] = float(np.mean(mfccs[i - 1]))
        features[f'mfcc_{i}_std']  = float(np.std(mfccs[i - 1]))

    return features


# Recopilar todos los formatos de audio compatibles con librosa
audio_extensions = ('*.wav', '*.flac', '*.mp3', '*.m4a', '*.ogg')
audio_files = []
for ext in audio_extensions:
    audio_files.extend(glob_module.glob(os.path.join(TEST_VOCES_DIR, ext)))
audio_files = sorted(audio_files)

print(f'Archivos de audio encontrados en test_voces/: {len(audio_files)}')

if len(audio_files) == 0:
    print(f'AVISO: No se encontraron audios en {TEST_VOCES_DIR}')
else:
    tv_features_list = []
    tv_names         = []

    for fp in audio_files:
        fname = os.path.basename(fp)
        print(f'  Procesando: {fname}')
        feats = extract_features_13(fp)
        if feats is not None:
            tv_features_list.append(feats)
            tv_names.append(fname)

    if not tv_features_list:
        print('ERROR: No se pudo extraer ninguna feature de test_voces/')
    else:
        df_tv = pd.DataFrame(tv_features_list)

        feats_faltantes_tv = [f for f in FINAL_FEATURES if f not in df_tv.columns]
        if feats_faltantes_tv:
            print(f'ERROR: Columnas faltantes: {feats_faltantes_tv}')
        else:
            X_tv    = df_tv[FINAL_FEATURES].values
            # Etiqueta real: todos los audios de test_voces/ son bonafide
            # (grabados por el equipo → voz humana real)
            y_tv_true = np.zeros(len(X_tv), dtype=int)   # 0 = bonafide
            y_tv_pred = model_final.predict(X_tv)

            n_total   = len(y_tv_true)
            n_correct = int(np.sum(y_tv_pred == y_tv_true))
            n_wrong   = n_total - n_correct

            print()
            print(f'Resultados — test_voces/ ({n_total} audios, todos bonafide reales):')
            print(f'  Clasificados como BONAFIDE (correcto): {n_correct}/{n_total} '
                  f'({100 * n_correct / n_total:.1f}%)')
            print(f'  Clasificados como SPOOF (error):       {n_wrong}/{n_total} '
                  f'({100 * n_wrong / n_total:.1f}%)')
            print()
            print('Detalle por archivo:')
            for fname, pred in zip(tv_names, y_tv_pred):
                estado = '\u2713 BONAFIDE' if pred == 0 else '\u2717 SPOOF (falso positivo)'
                print(f'  {estado:<28}  {fname}')

PARTE 1: Entrenamiento del Modelo Final (A01-A06 completos)
Modelo final entrenado: 5,160 muestras (2,580 spoof + 2,580 bonafide)
Distribución de ataques en train final:
attack_id
A01    446
A02    423
A03    441
A04    404
A05    425
A06    441
Name: count, dtype: int64

PARTE 2: Evaluación sobre dataset EVAL (ataques A07-A19)
AVISO: No se encontró ../../Obtencion_Metricas/dataset_caracteristicas_eval.csv
  Ejecuta primero obtencion_metricas_V1.ipynb (configurado para eval)
  y vuelve a correr esta celda.

PARTE 3: Evaluación sobre test_voces/ (voces reales del equipo)
Archivos de audio encontrados en test_voces/: 50
  Procesando: AUDIO-2026-04-28-16-00-52_16k.wav
  Procesando: AUDIO-2026-04-28-16-01-01_16k.wav
  Procesando: AlexisLiliana_VozEspañolWhatsiPhone.m4a
  Procesando: AlexisLiliana_VozEspañolWhatsiPhone_16k.wav
  Procesando: Alexis_VozEspañolMac.m4a
  AVISO: Error cargando Alexis_VozEspañolMac.m4a: 
  Procesando: Alexis_VozEspañolMac2.m4a
  AVISO: Error cargando Alexis_VozEs

# 📝 BLOC DE NOTAS — Justificación Teórica y Análisis de Resultados

---

## 1. Por qué Leave-One-Group-Out es superior al split aleatorio en ciberseguridad

En un **split aleatorio estratificado** (Modelos 1 y 2), tanto el conjunto de train como el de test contienen muestras de *todos* los algoritmos de ataque (A01–A06). Esto provoca una **fuga de distribución**: el modelo aprende la firma acústica de cada ataque y luego la reconoce en el test. Los resultados son optimistas y no reflejan la realidad operativa.

En **ciberseguridad**, el escenario real es el contrario: los sistemas defensivos se enfrentan continuamente a **nuevos ataques nunca vistos** durante el entrenamiento. Un spoofing-engine lanzado hoy con TTS de última generación no estará en ningún conjunto de entrenamiento histórico.

**Leave-One-Group-Out CV (LOGOCV)** replica precisamente ese escenario:

| Aspecto | Split aleatorio | LOGOCV |
|---------|----------------|--------|
| **¿Qué mide?** | Capacidad de memorización | Capacidad de **generalización** |
| **Fuga de información** | Sí (el test tiene ataques del train) | No (el test tiene ataques 100% nuevos) |
| **Validez en producción** | Baja | Alta |
| **Robustez estadística** | Un solo número | Media + desviación estándar sobre 6 folds |
| **Coste computacional** | Bajo | 6× mayor (un modelo por fold) |

Además, LOGOCV produce **6 estimaciones independientes** de rendimiento, lo que permite calcular una media y una desviación estándar estadísticamente informativas.

---

## 2. Cómo interpretar la desviación estándar del F1-Score

La desviación estándar entre folds **no mide el error del modelo** sino la **heterogeneidad entre algoritmos de ataque**. Su interpretación en el contexto de detección de deepfake audio es:

### STD baja (< 0.04)
- El modelo mantiene un rendimiento **consistente** independientemente del ataque excluido.
- Indica que el modelo ha capturado características acústicas **transversales** a todos los sistemas TTS/VC (e.g., artefactos de cuantización espectral comunes a múltiples vocoders).
- **Interpretación en ciberseguridad**: alta fiabilidad operativa. El sistema defensivo no tendrá comportamientos erráticos al enfrentarse a nuevos ataques.

### STD alta (> 0.08)
- El rendimiento **varía drásticamente** según qué ataque se excluye.
- El modelo aprendió características específicas de los 5 ataques en entrenamiento que no se transfieren al sexto.
- Hay ataques "fáciles" (bien cubiertos por los ataques de train) y ataques "difíciles" (con características acústicas fuera de la distribución entrenada).
- **Interpretación en ciberseguridad**: riesgo de puntos ciegos. Un adversario que lance un nuevo tipo de vocoder con arquitectura diferente podría evadir el sistema con alta probabilidad.

### STD moderada (0.04 – 0.08)
- Situación intermedia. El modelo generaliza razonablemente, pero algunos ataques presentan características más difíciles de extrapolar.
- Indica la necesidad de **re-entrenamiento periódico** con nuevos ataques para mantener la cobertura.

---

## 3. Conclusión: ¿Aprendió el modelo 'qué es una voz humana' o memorizó vulnerabilidades?

Esta es la pregunta central del experimento. Los tres escenarios posibles, con sus implicaciones:

### Escenario A — Generalización real (F1 medio alto + STD baja)
El modelo aprendió patrones acústicos **independientes del algoritmo generador**: ausencia de micro-variaciones prosódicas naturales, distribuciones espectrales inusuales en MFCCs, o correlaciones temporales que solo existen en voz sintética. Esto sugiere que los 13 MFCCs capturan algo cercano a la "firma de humanidad" de la voz.

**Implicación práctica**: el sistema es robusto y podría desplegarse en producción con confianza razonable.

### Escenario B — Generalización parcial (F1 medio moderado + STD moderada)
El modelo mezcla dos tipos de aprendizaje: generaliza sobre algunos ataques pero depende de las vulnerabilidades específicas de otros. Probablemente algunos ataques (e.g., A01-A02) son acústicamente similares entre sí, y el modelo aprendió a detectarlos entre sí, pero falla con los que son más innovadores.

**Implicación práctica**: el sistema funciona bien contra ataques conocidos pero requiere monitorización activa y re-entrenamiento ante nuevos sistemas de síntesis.

### Escenario C — Memorización (F1 medio bajo o STD muy alta)
El modelo aprendió las **vulnerabilidades específicas** de cada vocoder presente en el entrenamiento (artefactos del Griffin-Lim, del WaveNet, etc.) pero no generalizó a características universales de voz sintética. En ausencia del ataque conocido, el modelo falla.

**Implicación práctica**: el sistema es esencialmente un detector de firmas específicas, no un detector de "falsedad". Sería vulnerable ante cualquier nuevo sistema TTS de alta calidad.

---

### Nota metodológica

La evaluación sobre `test_voces/` añade una dimensión crítica: si el modelo clasifica correctamente como bonafide las voces reales del equipo, confirma que no está sesgado hacia predecir "todo es spoof" cuando las métricas de F1 son altas. Un sistema con recall alto pero baja especificidad en voces reales sería inutilizable en producción por la tasa de falsos positivos.

El resultado combinado de LOGOCV (generalización ante ataques desconocidos) + evaluación en `test_voces/` (no rechazar voces legítimas) constituye la evaluación más completa y realista posible con los datos disponibles.